# FULL-SYN — vòng 1

[Plan FULL-SYN](../../docs/project/full-syn-train-plan-2026-10-22.md), mục 4.1 và 5. Notebook này là **log của vòng 1**:
cell output ghi cấu hình, hash, kết quả từng trạm và quyết định ở mỗi cổng; log đầy đủ của code
(checkpoint, `train.log`, `step_metrics.jsonl`, predictions, policy, bản notebook đã chạy) nằm trong campaign
dưới `data/training/`. Vòng chỉnh sau là **file mới** (`05_full_syn_r2.ipynb`) với campaign và settings mới,
để giữ lịch sử đối chứng. Bị ngắt giữa chừng thì chạy lại chính file này: bước đã xong được bỏ qua,
task đang dở chạy tiếp từ `last.ckpt`.

Các cổng tự dừng notebook (cell báo `GateStop`):

| Cổng | Dừng khi |
|---|---|
| Sau G1 (D18) | val cAcc-M2 của FSA < 52,8% |
| Sau G2 (D17, D19) | 🟡/🔴 ở vòng 1–2; vòng 3: 🔴 (🟡 chạy tiếp, ghi "kỳ vọng ngang B2") |
| Trước khi train task 2 | precision nhãn giả của model task 1 trên ảnh task 2 < 0,5 |
| G4 | chỉ chấm test một lần; đã có kết quả thì in lại, không chấm lại |

In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_r1.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'fsa_batch_size', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

In [ ]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

## G1 — FSA và kiểm tra sớm (D18)

Fine-tune toàn bộ trên ảnh ghép task 1, tối đa `fsa_epochs`; xuất `hf_model` từ epoch có AP50 val_task_1 cao nhất.

In [ ]:
FSA = full_syn.stage_fsa(PLAN)
print('Epoch đã chọn:', FSA['epoch_selection']['epoch'], '| AP50 val:', [round(c['AP50'], 4) for c in FSA['epoch_selection']['curve']])
print(json.dumps({k: FSA[k] for k in ('cAcc_m2', 'cAcc_m1_single', 'AP50', 'cAcc_by_level', 'train_hours', 'gate')}, indent=1))

## G2 — PDP task 1 từ FSA và cổng D17

In [ ]:
TASK1 = full_syn.stage_task1(PLAN)
print(json.dumps({k: TASK1[k] for k in ('cAcc_m2', 'cAcc_m1_single', 'threshold_single', 'AP50', 'cAcc_by_level',
                                        'train_hours', 'gate')}, indent=1))
print('Nhầm lẫn nhiều nhất:', TASK1['confusions_top20'][:10])

## G3 — task 2–5

Trước mỗi task t: audit nhãn giả của model task t−1 trên `audit_images` ảnh ghép của task t, so với
`train_task_<t>_gt_full.json`. Sau mỗi task: chấm val, ngưỡng theo nhóm task, ghi `task_<t>/task_log.json`.

In [ ]:
TASKS = full_syn.stage_tasks(PLAN)
for t, r in sorted(TASKS['tasks'].items(), key=lambda item: int(item[0])):
    p = r['pseudo_labels']
    print(f"task {t}: cAcc M1 {r['cAcc_m1_groups']:.4f} (nhóm {r['thresholds_groups']}) / {r['cAcc_m1_single']:.4f} (một ngưỡng)"
          f" | AP50 {r['AP50']} | nhãn giả P {p['all'].get('precision')} R {p['recall']} | I3/F14 {p['filters']}"
          f" | {r['train_hours']} h")

## G4 — chấm test một lần

In [ ]:
FINAL = full_syn.stage_final_test(PLAN)
print((Path(PLAN['root']) / 'SUMMARY.md').read_text())